# ETL da camada silver para camada gold


## Extract

In [1]:
import pandas as pd
import numpy as np

data_layer_filepath = '../../data_layer/'

df = pd.read_csv(data_layer_filepath + 'silver/airbnb-dataset-silver.csv', low_memory=False)
print("Dataset carregado com sucesso!")
df.head()

Dataset carregado com sucesso!


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,1001254,Clean & quiet apt home by the park,80014485718,False,Madaline,Brooklyn,Kensington,40.64749,-73.97237,False,...,193.0,10,9,2021-10-19,0.21,4.0,6,286,Clean up and treat the home the way you'd like...,True
1,1002102,Skylit Midtown Castle,52335172823,True,Jenna,Manhattan,Midtown,40.75362,-73.98377,False,...,28.0,30,45,2022-05-21,0.38,4.0,2,228,Pet friendly but please confirm with me if the...,True
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,True,Elise,Manhattan,Harlem,40.80902,-73.94190,True,...,124.0,3,0,NaN,NaN,5.0,1,352,"I encourage you to use my kitchen, cooking and...",True
3,1002755,Sem nome informado,85098326012,False,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,True,...,74.0,30,270,2019-07-05,4.64,4.0,1,322,NaN,False
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,True,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,False,...,41.0,10,9,2018-11-19,0.10,3.0,1,289,"Please no smoking in the house, porch or on th...",True


## Transform

In [2]:
cols_fato = [
    'id',
    'name',
    'room_type',
    'minimum_nights',
    'cancellation_policy',
    'instant_bookable',
    'availability_365',
    'has_house_rules',
    'construction_year'
]

df_fato = df[cols_fato]
df_fato

,id,name,room_type,minimum_nights,cancellation_policy,instant_bookable,availability_365,has_house_rules,construction_year
0,1001254,Clean & quiet apt home by the park,Private room,10,strict,False,286,True,2020
1,1002102,Skylit Midtown Castle,Entire home/apt,30,moderate,False,228,True,2007
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,Private room,3,flexible,True,352,True,2005
3,1002755,Sem nome informado,Entire home/apt,30,moderate,True,322,False,2005
4,1003689,Entire Apt: Spacious Studio/Loft by central park,Entire home/apt,10,moderate,False,289,True,2009
...,...,...,...,...,...,...,...,...,...
99444,57358028,"Room in Queens, NY, near LGA.",Private room,1,strict,True,361,True,2022
99445,57358580,Cozy home away from home,Private room,1,moderate,True,324,False,2020
99446,57359133,Central Park Views - Private Room & Bathroom,Private room,1,strict,False,0,True,2012
99447,57359685,Ultimate 50th Floor Downtown Penthouse - 4000...,Entire home/apt,2,flexible,False,343,True,2020


In [7]:
df.dtypes

id                                  int64
name                               object
host_id                             int64
host_identity_verified               bool
host_name                          object
neighbourhood_group                object
neighbourhood                      object
lat                               float64
long                              float64
instant_bookable                     bool
cancellation_policy                object
room_type                          object
construction_year                   int64
price                             float64
service_fee                       float64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
review_rate_number                float64
calculated_host_listings_count      int64
availability_365                    int64
house_rules                        object
has_house_rules                   

In [10]:
df['neighbourhood'].isna().sum()

np.int64(0)

In [4]:
cols_preco = ['price', 'service_fee', 'id']
df_preco = df[cols_preco]
df_preco

,price,service_fee,id
0,966.0,193.0,1001254
1,142.0,28.0,1002102
2,620.0,124.0,1002403
3,368.0,74.0,1002755
4,204.0,41.0,1003689
...,...,...,...
99444,982.0,196.0,57358028
99445,946.0,189.0,57358580
99446,706.0,141.0,57359133
99447,1043.0,209.0,57359685


In [5]:
cols_avaliacao = [
    'id',
    'last_review',
    'reviews_per_month',
    'number_of_reviews',
    'review_rate_number'
]
df_avaliacao = df[cols_avaliacao]

df_avaliacao

,id,last_review,reviews_per_month,number_of_reviews,review_rate_number
0,1001254,2021-10-19,0.21,9,4.0
1,1002102,2022-05-21,0.38,45,4.0
2,1002403,NaN,NaN,0,5.0
3,1002755,2019-07-05,4.64,270,4.0
4,1003689,2018-11-19,0.10,9,3.0
...,...,...,...,...,...
99444,57358028,2019-06-29,8.58,239,2.0
99445,57358580,2019-06-27,2.84,76,1.0
99446,57359133,2017-08-15,0.14,4,4.0
99447,57359685,2019-07-01,0.74,21,4.0


In [11]:
cols_host = [
    'id',
    'host_id',
    'host_name',
    'host_identity_verified',
    'calculated_host_listings_count'
]
df_host = df[cols_host]

df_host

,id,host_id,host_name,host_identity_verified,calculated_host_listings_count
0,1001254,80014485718,Madaline,False,6
1,1002102,52335172823,Jenna,True,2
2,1002403,78829239556,Elise,True,1
3,1002755,85098326012,Garry,False,1
4,1003689,92037596077,Lyndon,True,1
...,...,...,...,...,...
99444,57358028,56457739998,Sonia,True,2
99445,57358580,60176837202,Sem nome informado,True,1
99446,57359133,68411243647,Sem nome informado,True,1
99447,57359685,95625271612,Sem nome informado,True,2


In [16]:
df.loc[df['name'] == '#NAME?']

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
9529,6664620,#NAME?,56170040879,True,John,Brooklyn,Williamsburg,40.71631,-73.96353,True,...,131.0,2,121,2019-06-18,2.55,4.0,8,48,NaN,False
9534,6667382,#NAME?,32946000035,False,John,Brooklyn,Williamsburg,40.71692,-73.96353,True,...,69.0,2,97,2019-06-15,2.07,4.0,8,54,NaN,False
9545,6673457,#NAME?,73912716130,False,John,Brooklyn,Williamsburg,40.71767,-73.96252,True,...,146.0,2,92,2019-06-08,1.94,3.0,8,134,Please remember that this is a residential bui...,True
9568,6686712,#NAME?,95379724186,False,John,Brooklyn,Williamsburg,40.71714,-73.96234,False,...,168.0,2,107,2019-06-21,2.26,3.0,8,400,I have no time restraints on Guests. Come and ...,True
9578,6692235,#NAME?,84121768078,True,John,Brooklyn,Williamsburg,40.71636,-73.96246,False,...,148.0,2,97,2019-06-18,2.13,3.0,8,226,Be courteous to neighbors. No parties. No smok...,True
9596,6702729,#NAME?,39667397754,False,John,Brooklyn,Williamsburg,40.71544,-73.96211,True,...,105.0,2,99,2019-06-23,2.10,2.0,8,70,We're flexible on check-in/checkout times,True
9918,6886093,#NAME?,24164569686,True,John,Brooklyn,Williamsburg,40.71596,-73.96215,True,...,119.0,2,106,2019-06-23,2.29,5.0,8,148,NaN,False
10087,6981640,#NAME?,94931619391,False,Gordon M,Manhattan,Harlem,40.82323,-73.95494,False,...,240.0,2,84,2019-05-26,1.82,3.0,4,20,You may come and go as you please. No smoking...,True
11780,7979094,#NAME?,38649567232,False,Gordon M,Manhattan,Harlem,40.82195,-73.95373,True,...,175.0,2,24,2019-06-14,0.55,3.0,4,94,Please treat the apartment like it is your own...,True
26500,16274640,#NAME?,97982966203,False,Sybilla Michelle,Manhattan,Hell's Kitchen,40.75575,-73.99374,False,...,114.0,2,44,2019-06-30,2.26,4.0,3,133,NaN,False


## Load